In [1]:
import os
import sys
from dotenv import load_dotenv
from pathlib import Path
import datetime as dt

import pandas as pd
import pandas_datareader.data as pdr
import yfinance as yf
import numpy as np
import matplotlib.pyplot as plt
import mplfinance as mpf
import talib as ta
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"

In [4]:
sys.path.append("/home/jovyan/notebook")
from Modules.get_market_data import GetMarketData
get_market_data = GetMarketData(Path('/home/jovyan/data'))

### 7.1 自由度の高い可視化

In [5]:
#x = [1, 2, 3, 4, 5]
#y = [10, 20, 30, 40, 50]
#data = go.Bar(x=x, y=y)
#fig = go.Figure(data=data)
#fig.show()

#### 7.1.2 ローソク足チャートを表示する

In [16]:
name = "積水ハウス"
code = "1928.T"

# チャート表示開始日
display_start_date = dt.datetime(2021, 12, 1)
# データ取得終了日(現在の日付を取得)
end_date = dt.datetime(2022, 3, 31)
# データ取得開始日
start_date = end_date - dt.timedelta(days=300)

### 積水ハウス (1928.T)
df = get_market_data.get_data_from_yfinance(code, start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'))
if isinstance(df.columns, pd.MultiIndex):
    df = df.droplevel(level=1, axis=1)
close = df['Close']

# 移動平均線
df["ma5"], df["ma25"] = ta.SMA(close, timeperiod=5), ta.SMA(close, timeperiod=25)

rdf = df[display_start_date:end_date]

# インデクスを文字列型に
rdf.index = pd.to_datetime(rdf.index).strftime('%m-%d-%Y')

# レイアウト定義
layout = {
    "title": { "text": " {} {} ".format(code, name), "x": 0.5 },
    "xaxis": {"title": "日付", "rangeslider": {"visible": False}},
    "yaxis": {"title": "価格(円)", "side": "left", "tickformat": ","},
    "plot_bgcolor": "light blue",
}

# データ定義
data = [
    # ローソク足
    go.Candlestick(x=rdf.index, open=rdf['Open'], high=rdf['High'],
                   low=rdf['Low'], close=rdf['Close'],
                   increasing_line_color='red',
                   #increasing_line_width=2,
                   #increasing_fillcolor='red',
                   decreasing_line_color='gray'
                   #decreasing_line_width=2,
                   #decreasing_fillcolor='gray'
    ),
    # 5日移動平均線
    go.Scatter(x=rdf.index, y=rdf['ma5'], name='MA5',
                 line={ 'color': 'royalblue', 'width': 1.2 }),
    # 25日移動平均線
    go.Scatter(x=rdf.index, y=rdf['ma25'], mode='lines', name='MA25',
                 line={ 'color': 'lightgreen', 'width': 1.2 }),
    
]

# グラフ生成
fig = go.Figure(data=data, layout=go.Layout(layout))

# レイアウトを更新
fig.update_layout({
    "xaxis": {
        # 日付を半分に
        "tickvals": rdf.index[::2],
        # 表示形式をMM-DDに
        "ticktext": ["{}-{}".format(x.split('-')[0], x.split('-')[1]) for x in rdf.index[::2]]
    }
})

fig.show()

[*********************100%***********************]  1 of 1 completed

In [31]:

# データ取得終了日(現在の日付を取得)
end_date = dt.datetime(2022, 4, 1)

# チャート表示開始日(100日前の日付を計算)
display_start_date = dt.datetime(2021, 11, 1)

# データ取得開始日(300日前の日付を計算)
start_date = display_start_date - dt.timedelta(days=300)

### 日本製鉄 (5401.T)
name = "日本製鉄"
code = "5401.T"
df = get_market_data.get_data_from_yfinance(code, start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'))
if isinstance(df.columns, pd.MultiIndex):
    df = df.droplevel(level=1, axis=1)
close = df['Close']

# 移動平均線
df["ma5"], df["ma25"] = ta.SMA(close, timeperiod=5), ta.SMA(close, timeperiod=25)

# ゴールデンクロス
cross = df["ma5"] > df["ma25"]
df["cross"] = cross
cross_shift = df["cross"].shift(1)

# ゴールデンクロスの発生日
temp_gc = (cross != cross_shift) & (cross == True)

# デッドクロスの発生日
temp_dc = (cross != cross_shift) & (cross == False)

# ゴールデンクロスの発生日であればMA5の値、それ以外はNaN
gc = [m if g == True else np.nan for g, m in zip(temp_gc, df["ma5"])]

# デッドクロスの発生日であればMA25の値、それ以外はNaN
dc = [m if d == True else np.nan for d, m in zip(temp_dc, df["ma25"])]

# データフレームのカラムとして保存
df["gc"], df["dc"] = gc, dc

# MACD  シグナル, ヒストグラム
macd, macdsignal, hist = ta.MACD(close, fastperiod=12, slowperiod=26,
                                 signalperiod=9)
df['macd'] = macd
df['macd_signal'] = macdsignal
df['hist'] = hist

# RSI
rsi14 = ta.RSI(close, timeperiod=14)
rsi28 = ta.RSI(close, timeperiod=28)
df['rsi14'], df['rsi28'] = rsi14, rsi28

# 補助線
df['70'] = [70 for _ in close]
df['30'] = [30 for _ in close]

# ストキャスティクス
slowK, slowD = ta.STOCH(df['High'], df['Low'], df['Close'],
                        fastk_period=5, slowk_period=3, 
                        slowk_matype=0, slowd_period=3,
                        slowd_matype=0)
df['slowK'], df['slowD'] = slowK, slowD
df['80'] = [80 for _ in close]
df['20'] = [20 for _ in close]

# ボリンジャーバンド±2σ
df["upper2"], _, df["lower2"] = ta.BBANDS(close, timeperiod=25, nbdevup=2,
                                          nbdevdn=2, matype=ta.MA_Type.SMA)

rdf = df[display_start_date:end_date]

# インデクスを文字列型に
rdf.index = pd.to_datetime(rdf.index).strftime('%m-%d-%Y')

# レイアウト定義
layout = {
    "height": 1000,
    "title"  : { "text": "{} {}".format(code, name), "x":0.5 },
    "xaxis" : { "rangeslider": { "visible": False } },
    "yaxis1": { "domain": [.46, 1.0], "title": "価格（円）", "side": "left", "tickformat": "," },
    "yaxis2": { "domain": [.40, .46] },
    "yaxis3": { "domain": [.30, .395], "title": "MACD", "side": "right"}, # MACD
    "yaxis4": { "domain": [.20, .295], "title": "RSI", "side": "right"}, # RSI
    "yaxis5": { "domain": [.10, .195], "title": "STC", "side": "right"}, # ストキャスティクス
    "yaxis6": { "domain": [.00, .095], "title": "Volume", "side": "right"}, # 出来高
    "plot_bgcolor":"light blue"
}

data = [
    # ローソク足
    go.Candlestick(yaxis='y', x=rdf.index, open=rdf['Open'], 
                   high=rdf['High'], low=rdf['Low'], close=rdf['Close'],
                   increasing_line_color='red', decreasing_line_color='gray'),
    # 5日移動平均線
    go.Scatter(yaxis='y', x=rdf.index, y=rdf["ma5"], name='MA5',
                 line={ 'color': 'royalblue', 'width': 1.2 }),
    # 25日移動平均線
    go.Scatter(yaxis='y', x=rdf.index, y=rdf['ma25'], name='MA25',
                 line={ 'color': 'lightgreen', 'width': 1.2 }),
    # ゴールデンクロス
    go.Scatter(yaxis='y', x=rdf.index, y=rdf['gc'],
                name='Golden Cross',
                opacity=0.5, mode='markers',
                marker={ 'size': 15, 'color': 'purple' }),
    # デッドクロス
    go.Scatter(yaxis='y', x=rdf.index, y=rdf['dc'],
                name='Dead Cross',
                opacity=0.5, mode='markers',
                marker={ 'size': 15, 'color': 'black', 'symbol': 'x' }),
    # ボリンジャーバンド
    go.Scatter(yaxis='y', x=rdf.index, y=rdf['upper2'], name='',
                 line={ 'color': 'lavender', 'width': 0 }),
    go.Scatter(yaxis='y', x=rdf.index, y=rdf['lower2'],
                name='BB', line={ 'color': 'lavender', 'width': 0 },
                fill='tonexty', fillcolor="rgba(170, 170, 255, 0.2)"),
    # MACD
    go.Scatter(yaxis='y3', x=rdf.index, y=rdf['macd'],
                name='MACD', line={ 'color': 'magenta', 'width': 1 }),
    go.Scatter(yaxis='y3', x=rdf.index, y=rdf['macd_signal'],
                name='MACD Signal', line={ 'color': 'green', 'width': 1 }),
    go.Bar(yaxis='y3', x=rdf.index, y=rdf['hist'], 
                name='histogram', marker={  'color': 'slategray' } ),
    # RSI
    go.Scatter(yaxis='y4', x=rdf.index, y=rdf['rsi14'],
                name='RSI14',
                line={ 'color': 'magenta', 'width': 1. }),
    go.Scatter(yaxis='y4', x=rdf.index, y=rdf['rsi28'],
                name='RSI28',
                line={ 'color': 'green', 'width': 1 }),
    # 補助線
    go.Scatter(yaxis='y4', x=rdf.index, y=rdf['70'], mode='lines', name='70',
                 line={ 'color': 'black', 'width': 1.0 }),
    go.Scatter(yaxis='y4', x=rdf.index, y=rdf['30'], mode='lines', name='30',
                 line={ 'color': 'red', 'width': 1.0 }),
    # ストキャスティクス
    go.Scatter(yaxis='y5', x=rdf.index, y=rdf['slowK'], name='SlowK',
                 line={ 'color': 'black', 'width': 1.2 }),
    go.Scatter(yaxis='y5', x=rdf.index, y=rdf['slowD'], name='SlowD',
                 line={ 'color': 'black', 'width': 1.2 }),
    # 補助線
    go.Scatter(yaxis='y5', x=rdf.index, y=rdf['20'], name='20',
                 line={ 'color': 'blue', 'width': 1.0, 'dash': 'dot' }),
    go.Scatter(yaxis='y5', x=rdf.index, y=rdf['80'], name='80',
                 line={ 'color': 'blue', 'width': 1.0, 'dash': 'dot' }),
    # 出来高
    go.Bar(yaxis='y6', x=rdf.index, y=rdf['Volume'], name='Volume', 
            marker={ 'color': 'lightgray' } )
]

# グラフ生成
fig = go.Figure(data=data, layout=go.Layout(layout))

# レイアウトを更新
fig.update_layout({
    "xaxis": {
        # 日付を半分に
        "tickvals": rdf.index[::2],
        # 表示形式をMM-DDに
        "ticktext": ["{}-{}".format(x.split('-')[0], x.split('-')[1]) for x in rdf.index[::2]]
    }
})

fig.show()

[*********************100%***********************]  1 of 1 completed